<br><br><br>
# Pytorch Embeddings Module
<br>

## Advanced Notebook
<br>

This notebook is meant to illustrate the advanced features of the **pytorch_embeddings** python module. 

### Heuristic

First, let's import the modules we need. 

In [1]:
import torch
import pytorch_embeddings

<br>

In the cell below we create two Frobenius embeddings, with various *frobenius_rank* and *frobenius_blocks* parameters. <br>
Observe, that the sizes of these embeddings are different. Moreover, notice that the embedding dimension is 512. <br>
This seems to contradict our previous claim (in the basic notebook) about the embedding dimensions Frobenius embeddings can work with. 
<br><br>

In [2]:
num_embeddings = 10240
embedding_dim = 512

embedding1 = pytorch_embeddings.FrobeniusEmbedding(num_embeddings, embedding_dim, frobenius_rank=8, frobenius_blocks=2)
embedding2 = pytorch_embeddings.FrobeniusEmbedding(num_embeddings, embedding_dim, frobenius_rank=32, frobenius_blocks=4)

print(embedding1)
print(embedding2)

Embedding(embedding_type=frobenius, num_embeddings=10240, embedding_dim=512, precision=fp32, size=0.007202 MB, frobenius_rank=8, frobenius_blocks=2)
Embedding(embedding_type=frobenius, num_embeddings=10240, embedding_dim=512, precision=fp32, size=0.04199 MB, frobenius_rank=32, frobenius_blocks=4)


<br>

However, if we tried to run the below command: 

*pytorch_embeddings.FrobeniusEmbedding(num_embeddings, embedding_dim, frobenius_rank=8, frobenius_blocks=1)*

it would result in an "invalid parameters" error. <br>
Whether the parameters of a Frobenius embedding are valid depends on a number of conditions, including the hardware architecture of the GPU we intend to use it on. <br>
The function below works as an acceptable filter to help us find the valid parameter configurations. <br><br>

In [3]:
embedding_dims_supported = [8, 16, 32, 64, 128, 256]


def valid_configuration(embedding_dim, frobenius_rank, frobenius_blocks):
    condition1 = frobenius_rank % 8 == 0
    condition2 = embedding_dim % frobenius_blocks == 0
    condition3 = (embedding_dim // frobenius_blocks) in embedding_dims_supported
    conditions = [condition1, condition2, condition3]
    return all(conditions)

<br><br>
Let's test it! 
<br><br>

In [4]:
print(valid_configuration(512, 8, 1))
print(valid_configuration(512, 32, 4))

False
True


<br><br>
Rather than to brute force our way through all the valid configurations, the **pytorch_embeddings** module offers a simple heuristic, which recommends parameters for us. 
<br><br>

In [5]:
num_embeddings = 10240
embedding_dim = 128

pytorch_embeddings.heuristic('frobenius', num_embeddings, embedding_dim, 'fp32')

[config(embedding_type='frobenius', num_embeddings=10240, embedding_dim=128, size=21504, precision='fp32', frobenius_rank=16, frobenius_blocks=1),
 config(embedding_type='frobenius', num_embeddings=10240, embedding_dim=128, size=17408, precision='fp32', frobenius_rank=8, frobenius_blocks=2)]

<br><br>
In the above example, the heuristic recommended two different configurations for us to try. These come with different accuracy, latency and throughput. <br><br>
As a rule of thumb, we expect *frobenius_blocks* to affect both accuracy and latency to a larger extent than *frobenius_rank* does. <br>

- increasing *frobenius_rank* will improve the capacity of the model - but it will also make the job of the optimizer more difficult. 
- increasing *frobenius_blocks* is expected to make the job of the optimizer easier, while also improving the capacity of the model. <br>
However, it will also significantly increase the size of the model and incur a larger latency hit. <br>

It must be noted, that when we create a new Frobenius embedding with *frobenius_rank* and *frobenius_blocks* unset, then by default a heuristic will select one of the recommendations of **pytorch_embeddings.heuristic**. 
<br><br>

In [6]:
pytorch_embeddings.FrobeniusEmbedding(num_embeddings, embedding_dim)

Embedding(embedding_type=frobenius, num_embeddings=10240, embedding_dim=128, precision=fp32, size=0.004272 MB, frobenius_rank=8, frobenius_blocks=2)

<br><br>
Naturally, both *frobenius_rank* and *frobenius_blocks* parameters can be set through our context manager. 
<br><br>

In [7]:
from pytorch_embeddings import decomposed_embeddings

In [8]:
with decomposed_embeddings(embedding_type=['frobenius','frobenius','native'], frobenius_rank=[16,8], frobenius_blocks=[1,2]):
    embedding_A = torch.nn.Embedding(num_embeddings, embedding_dim, device='cuda')
    embedding_B = torch.nn.Embedding(num_embeddings, embedding_dim, device='cuda')
    embedding_C = torch.nn.Embedding(num_embeddings, embedding_dim, device='cuda')

print(embedding_A)
print(embedding_B)
print(embedding_C)

Embedding(embedding_type=frobenius, num_embeddings=10240, embedding_dim=128, precision=fp32, size=0.005249 MB, frobenius_rank=16, frobenius_blocks=1)
Embedding(embedding_type=frobenius, num_embeddings=10240, embedding_dim=128, precision=fp32, size=0.004272 MB, frobenius_rank=8, frobenius_blocks=2)
Embedding(embedding_type=native, num_embeddings=10240, embedding_dim=128, precision=fp32, size=5.0 MB)


<br>

### How to run it on the CPU

<br><br>
There is only one way to make Frobenius embeddings run on the **CPU**. <br>
The *onchip_memory* flag has to be set to **False**. Moreover, the embedding has to be constructed on the **CPU**. 
<br><br>

In [9]:
num_embeddings = 10240
embedding_dim = 512


embedding = pytorch_embeddings.FrobeniusEmbedding(num_embeddings, embedding_dim, device='cpu', onchip_memory=False)
embedding.uniform_(-1.0,1.0)

print(embedding)
print('device of weight:', embedding.weight.device)

Embedding(embedding_type=frobenius, num_embeddings=10240, embedding_dim=512, precision=fp32, size=0.007202 MB, frobenius_rank=8, frobenius_blocks=2)
device of weight: cpu


In [10]:
index = torch.randint(0,num_embeddings,(1,),device='cpu')
print('device of input:', index.device)
print('device of output:', embedding(index).device)

device of input: cpu
device of output: cpu


<br><br>
The primary function of the *onchip_memory* flag is to select the algorithm in the forward pass. <br> 
When it is set to **True**, the algorithm with lower memory footprint will be used. <br>
This however, can be expected to come at a small latency hit. <br> 
The corresponding algorithm for the **CPU** was not implemented, and so we must set it to **False** here.  
<br><br>

### Deployment

<br><br>
Frobenius embeddings can be deployed into Torchscript, ONNX, and then into TensorRT. <br>
However, at the moment, they can be run only in Torchscript or TensoRT. They cannot be run in ONNXRuntime. We might remedy this later. <br>
To see how the deployment process goes, let's make an example! <br><br>

In [11]:
import torch
import pytorch_embeddings
from pytorch_embeddings import decomposed_embeddings

In [12]:
batch_size = 1024
num_embeddings = 10240
embedding_dim = 32


with decomposed_embeddings(embedding_type=['frobenius'],frobenius_rank=[8],frobenius_blocks=[1]):
    model = torch.nn.Embedding(num_embeddings, embedding_dim)
    torch.nn.init.normal_(model.weight, 0.0, 1.0)

# better not forget to set your model into eval mode before deployment! 
model.eval()

index = torch.randint(0,num_embeddings,(batch_size,),dtype=torch.int64,device='cuda')
model_traced = torch.jit.trace(model, index)

In [13]:
model(index)

tensor([[ 0.0100, -0.4727,  0.3870,  ..., -0.6252,  0.1861, -0.1207],
        [ 2.0946, -0.3542,  0.9546,  ..., -0.0699, -0.0287, -0.0763],
        [-0.5662, -0.9369, -0.5073,  ..., -1.5646,  0.4774,  0.2476],
        ...,
        [ 1.2180, -0.3396,  0.8507,  ...,  0.3749, -0.2596, -0.2639],
        [ 0.3428, -0.3474,  0.6666,  ..., -0.5359, -0.1426, -0.3450],
        [-0.4832,  0.8038,  0.3412,  ...,  0.0617,  0.7787, -0.2226]],
       device='cuda:0', grad_fn=<CppNode<FrobeniusFunction>>)

In [14]:
# warmup and recompilation of torchscript models may take too long to run.... 
# especially for large models
# here we just turn off graph optimizations to deal with that
with torch.jit.optimized_execution(False):
    output = model_traced(index)
output

tensor([[ 0.0100, -0.4727,  0.3870,  ..., -0.6252,  0.1861, -0.1207],
        [ 2.0946, -0.3542,  0.9546,  ..., -0.0699, -0.0287, -0.0763],
        [-0.5662, -0.9369, -0.5073,  ..., -1.5646,  0.4774,  0.2476],
        ...,
        [ 1.2180, -0.3396,  0.8507,  ...,  0.3749, -0.2596, -0.2639],
        [ 0.3428, -0.3474,  0.6666,  ..., -0.5359, -0.1426, -0.3450],
        [-0.4832,  0.8038,  0.3412,  ...,  0.0617,  0.7787, -0.2226]],
       device='cuda:0', grad_fn=<CppNode<FrobeniusFunction>>)

<br><br>
That's it! Above, **embedding_traced** is already in Torchscript format! <br><br>
From converting to Torchscript, some latency improvement can be expected. <br>
However, we get the best performance on the **GPU** by converting our model to TensorRT format! <br>
First, let's try to convert it to ONNX. In order to do this, we need to export the **traced** model:
<br><br>

In [15]:
onnx_model_path = "model.onnx"


# here we turn off warmup and recompilations to make onnx tracing finish quickly
# + better make sure to set verbose to False for large models! 
with torch.jit.optimized_execution(False):
    torch.onnx.export(model_traced, index, onnx_model_path, verbose=False, opset_version=16, export_params=True, 
                      keep_initializers_as_inputs=False, custom_opsets={"trt.plugins" : 1}, 
                      do_constant_folding=True, input_names=['index'], output_names=['output'], 
                      dynamic_axes={'index' : {0 : 'batch_size'}, 'output' : {0 : 'batch_size'}})

/usr/local/lib/python3.10/dist-packages/torch/onnx/utils.py:847: UserWarning: no signature found for <torch.ScriptMethod object at 0x7f7103936390>, skipping _decide_input_format
  warnings.warn(f"{e}, skipping _decide_input_format")


<br><br>
The above command exports the model to "model.onnx". The *opset_version* parameter depends on the docker container version we use. <br>
The appropriate symbolic onnx operation is registered at the time we import the **pytorch_embeddings** module. <br><br>
Now, it's time to upload the "model.onnx" file to the machine we intend to do inference on. <br>
Once it's on the target machine, we may convert it to TensorRT as follows. 
<br><br>

In [16]:
# we use trtexec to convert the ONNX model into a TensorRT engine
import os
import subprocess


# first, we obtain the path to our custom TensorRT plugin
package_path = os.path.dirname(os.path.realpath(pytorch_embeddings.__file__))
plugin_path = os.path.join(package_path, 'frobenius_operator', 'tensorrt', 'libfrobenius_operatorPlugin.so')

# trt_engine_path sets the file we will save our TensorRT engine into
trt_engine_path = "model.engine"

# now we export a sample input into a file
sample_path = "index.dat"
index = torch.randint(0,num_embeddings,(batch_size,),dtype=torch.int64)
index = index.to(device='cpu', dtype=torch.int32).numpy()    # TensorRT needs the inputs to be in int32 format! 
index.tofile(sample_path)


# let's define the functions we need for the ONNX->TensorRT conversion
def execute(command):
    command = command.split()
    outputs = []
    stdout = subprocess.PIPE
    with subprocess.Popen(command, stdout=stdout, bufsize=1, 
                       universal_newlines=True) as process:
        for line in process.stdout:
            line = line[:-1]
            outputs.append(line)
            print(line)
    output = ''.join(outputs)
    return output


def shape_to_str(shape):
    return "x".join([str(i) for i in shape])


def convert_to_tensorrt(precision, sample_path):
    command = "trtexec --onnx=" + onnx_model_path
    # the flag below loads our tensorrt plugin
    command += " --staticPlugins=" + plugin_path
    # the name of the input was set to `index` at the time
    # we called `torch.onnx.export`
    command += " --loadInputs=index:" + sample_path
    command += " --inputIOFormats=int32:chw"
    if precision == 'fp16':
        command += " --outputIOFormats=fp16:chw"
    else:
        command += " --outputIOFormats=fp32:chw"
    if precision == 'fp16':
        command += " --fp16"
    min_shapes = "index:" + shape_to_str(index.shape)
    command += " --minShapes=" + min_shapes
    opt_shapes = "index:" + shape_to_str(index.shape)
    command += " --optShapes=" + opt_shapes
    max_shapes = "index:" + shape_to_str(index.shape)
    command += " --maxShapes=" + max_shapes
    # we don't measure the time of the data transfers from and to the GPU
    command += " --noDataTransfers"
    # this can make the performance measurements more accurate
    command += " --useSpinWait"
    # ideally, we'd like to optimize the graph as much as possible
    # so we set builderOptimizationLevel to 5 (the maximum)
    command += " --builderOptimizationLevel=5"
    # it is a good idea to set maxAuxStreams to the number of 
    # Frobenius embeddings in the model, 
    # so that they can be executed in parallel
    command += " --maxAuxStreams=1"
    # we give the builder a sufficiently large
    # workspace to try heuristics with - 
    # here we have set it to 1024 MiB
    command += " --memPoolSize=workspace:1024"
    command += " --saveEngine=" + trt_engine_path
    command += " --iterations=10000"
    command += " --avgRuns=10000"
    command += " --warmUp=1000"
    # the next line skips inference
    # command += " --skipInference"
    execute(command)


convert_to_tensorrt('fp32', sample_path)

&&&& RUNNING TensorRT.trtexec [TensorRT v8601] # trtexec --onnx=model.onnx --staticPlugins=/usr/local/lib/python3.10/dist-packages/pytorch_embeddings/frobenius_operator/tensorrt/libfrobenius_operatorPlugin.so --loadInputs=index:index.dat --inputIOFormats=int32:chw --outputIOFormats=fp32:chw --minShapes=index:1024 --optShapes=index:1024 --maxShapes=index:1024 --noDataTransfers --useSpinWait --builderOptimizationLevel=5 --maxAuxStreams=1 --memPoolSize=workspace:1024 --saveEngine=model.engine --iterations=10000 --avgRuns=10000 --warmUp=1000
[11/07/2023-08:22:25] [I] === Model Options ===
[11/07/2023-08:22:25] [I] Format: ONNX
[11/07/2023-08:22:25] [I] Model: model.onnx
[11/07/2023-08:22:25] [I] Output:
[11/07/2023-08:22:25] [I] === Build Options ===
[11/07/2023-08:22:25] [I] Max batch: explicit batch
[11/07/2023-08:22:25] [I] Memory Pools: workspace: 1024 MiB, dlaSRAM: default, dlaLocalDRAM: default, dlaGlobalDRAM: default
[11/07/2023-08:22:25] [I] minTiming: 1
[11/07/2023-08:22:25] [I] a

[11/07/2023-08:22:34] [W] * GPU compute time is unstable, with coefficient of variance = 324.645%.
[11/07/2023-08:22:34] [W]   If not already in use, locking GPU clock frequency or adding --useSpinWait may improve the stability.


<br><br>
That concludes the advanced notebook! <br>
The reader may wish to check the accuracy and performance of the TensorRT engine. <br>
The code that does that can be found in our example repository. <br><br>